# ⚽ FIFA World Cup 2026 — End-to-End ML Analysis

> **Author:** Shubham Indulkar  
> **Data:** martj42 international results · StatsBomb Open Data  
> **Model:** XGBoost + ELO ratings + rolling form + H2H features  
> **Goal:** Predict Win / Draw / Loss probabilities for WC 2026 group stage fixtures

---

## Table of Contents
1. [Setup & Data Download](#1)
2. [Exploratory Data Analysis](#2)
3. [ELO Rating System](#3)
4. [Feature Engineering](#4)
5. [Match Outcome Model (XGBoost)](#5)
6. [WC 2026 Group Stage Predictions](#6)
7. [xG-Based Match Simulation](#7)
8. [Penalty Shootout Simulator](#8)
9. [Key Findings](#9)

---

> **Note on Transfermarkt squad values:** Squad market values were sourced separately from  
> [transfermarkt-datasets](https://github.com/dcaribou/transfermarkt-datasets) — download  
> `data/prep/player_valuations.csv` and join on team + year. Squad values are **not required**  
> for this notebook (ELO + form features are the most predictive anyway), but they improve  
> calibration for mismatched-strength fixtures. See the main repo's `src/transfermarkt_client.py`  
> for the full integration.


## 1. Setup & Data Download <a id='1'></a>

In [ ]:
# Install non-standard packages (most are pre-installed on Kaggle)
import subprocess, sys
for pkg in ['xgboost', 'shap', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All packages ready.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import math
import time
from collections import defaultdict, deque
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import shap

# Working directory (Kaggle writes here)
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

print(f'Working dir: {WORK_DIR}')
print(f'NumPy {np.__version__}, Pandas {pd.__version__}')

In [ ]:
# ── Download martj42 international results ──────────────────────────────────
MARTJ42_BASE = 'https://raw.githubusercontent.com/martj42/international_results/master'

def download_csv(url: str, path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f'Downloading {url} ...')
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        path.write_bytes(r.content)
    return pd.read_csv(path, low_memory=False)

results_path = WORK_DIR / 'results.csv'
goalscorers_path = WORK_DIR / 'goalscorers.csv'

results_raw = download_csv(f'{MARTJ42_BASE}/results.csv', results_path)
goalscorers_raw = download_csv(f'{MARTJ42_BASE}/goalscorers.csv', goalscorers_path)

print(f'Results: {len(results_raw):,} rows | Goalscorers: {len(goalscorers_raw):,} rows')

In [ ]:
# ── Team normalisation map ───────────────────────────────────────────────────
_ALIASES = {
    'Korea Republic': 'South Korea', 'Republic of Korea': 'South Korea',
    'Korea, South': 'South Korea', 'IR Iran': 'Iran', 'USA': 'United States',
    "Côte d'Ivoire": 'Ivory Coast', "Cote d'Ivoire": 'Ivory Coast',
    'Cote d Ivoire': 'Ivory Coast', 'Czech Republic': 'Czechia',
    'FYR Macedonia': 'North Macedonia', 'Macedonia': 'North Macedonia',
    'Congo DR': 'DR Congo', 'Democratic Republic of Congo': 'DR Congo',
    'Türkiye': 'Turkey', 'Turkiye': 'Turkey',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Republic of Ireland': 'Ireland', 'China PR': 'China',
    'Trinidad & Tobago': 'Trinidad and Tobago',
    'West Germany': 'Germany', 'East Germany': 'Germany',
    'Yugoslavia': 'Serbia', 'Serbia and Montenegro': 'Serbia',
    'USSR': 'Russia',
}

def norm(name: str) -> str:
    return _ALIASES.get(str(name).strip(), str(name).strip())

# ── Pre-process results ──────────────────────────────────────────────────────
INCLUDE_KW = ['world cup','euro','european championship','friendly','qualif',
               'nations league','copa america','africa cup','asian cup','concacaf','gold cup']
EXCLUDE_KW = ['women','u21','u23','u20','u19','u18','olympic','youth','b team']

df = results_raw.copy()
df = df.rename(columns={'date': 'match_date'})
df['match_date'] = pd.to_datetime(df['match_date'], errors='coerce')
df = df.dropna(subset=['match_date', 'home_score', 'away_score'])
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)
df['home_team'] = df['home_team'].apply(norm)
df['away_team'] = df['away_team'].apply(norm)
df = df[df['match_date'] >= '2000-01-01']

t = df['tournament'].astype(str).str.lower()
df = df[t.apply(lambda x: any(k in x for k in INCLUDE_KW))]
df = df[~t.apply(lambda x: any(k in x for k in EXCLUDE_KW))]

df['match_id'] = (df['match_date'].dt.strftime('%Y%m%d') + '_' +
                   df['home_team'].str.replace(' ', '') + '_' +
                   df['away_team'].str.replace(' ', ''))
df = df.drop_duplicates('match_id')
df['is_neutral'] = df['neutral'].astype(str).str.lower().isin(['true', '1']).astype(int)
df['stage'] = 'group'

print(f'Clean match records: {len(df):,} | Date range: {df.match_date.min().date()} → {df.match_date.max().date()}')
df.head(3)

## 2. Exploratory Data Analysis <a id='2'></a>

In [ ]:
# Match counts by year + outcome distribution
df['year'] = df['match_date'].dt.year
df['total_goals'] = df['home_score'] + df['away_score']
df['result'] = df.apply(lambda r: 'Home Win' if r.home_score > r.away_score
                         else ('Away Win' if r.away_score > r.home_score else 'Draw'), axis=1)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('Matches per Year', 'Outcome Distribution', 'Goals per Match Distribution'))

year_counts = df.groupby('year').size().reset_index(name='count')
fig.add_trace(go.Bar(x=year_counts.year, y=year_counts['count'],
                      marker_color='steelblue', name='Matches'), row=1, col=1)

res_counts = df['result'].value_counts()
fig.add_trace(go.Pie(labels=res_counts.index, values=res_counts.values,
                      hole=0.4, name='Outcomes'), row=1, col=2)

fig.add_trace(go.Histogram(x=df['total_goals'], nbinsx=15,
                             marker_color='coral', name='Goals'), row=1, col=3)

fig.update_layout(title='📊 Men\'s International Football — 2000 to Present',
                   height=380, showlegend=False,
                   paper_bgcolor='#0e1117', plot_bgcolor='#0e1117',
                   font=dict(color='white'))
fig.show()

In [ ]:
# Top 15 teams by win rate (min 50 matches)
long = pd.concat([
    df.assign(team=df.home_team, win=(df.home_score > df.away_score).astype(int)),
    df.assign(team=df.away_team, win=(df.away_score > df.home_score).astype(int))
])
team_stats = long.groupby('team').agg(matches=('win','count'), wins=('win','sum')).reset_index()
team_stats['win_rate'] = team_stats.wins / team_stats.matches
top15 = team_stats[team_stats.matches >= 50].nlargest(15, 'win_rate')

fig = px.bar(top15, x='win_rate', y='team', orientation='h',
              text=top15['win_rate'].apply(lambda x: f'{x:.1%}'),
              color='win_rate', color_continuous_scale='Blues',
              title='🏆 Top 15 Nations by Win Rate (min. 50 matches, 2000–present)')
fig.update_layout(height=450, paper_bgcolor='#0e1117', plot_bgcolor='#0e1117',
                   font=dict(color='white'), coloraxis_showscale=False)
fig.update_traces(textposition='inside')
fig.show()

In [ ]:
# Home advantage breakdown by tournament type
def tcat(t):
    t = str(t).lower()
    if 'world cup' in t and 'qualif' not in t: return 'World Cup'
    if ('euro' in t or 'european' in t) and 'qualif' not in t: return 'Euro'
    if 'qualif' in t: return 'Qualifier'
    if 'friendly' in t: return 'Friendly'
    if 'nations league' in t: return 'Nations League'
    if 'copa america' in t: return 'Copa América'
    if 'africa cup' in t: return 'AFCON'
    return 'Other'

df['tcat'] = df['tournament'].apply(tcat)
ha = (df[df.is_neutral == 0]
      .assign(home_win=(df.home_score > df.away_score).astype(int))
      .groupby('tcat')
      .agg(home_win_rate=('home_win','mean'), n=('home_win','count'))
      .reset_index()
      .query('n > 30')
      .sort_values('home_win_rate', ascending=True))

fig = px.bar(ha, x='home_win_rate', y='tcat', orientation='h',
              text=ha['home_win_rate'].apply(lambda x: f'{x:.1%}'),
              color='home_win_rate', color_continuous_scale='RdYlGn',
              title='🏠 Home Win Rate by Tournament Type (non-neutral venues)')
fig.update_layout(height=380, paper_bgcolor='#0e1117', plot_bgcolor='#0e1117',
                   font=dict(color='white'), coloraxis_showscale=False)
fig.show()

## 3. ELO Rating System <a id='3'></a>

ELO ratings provide a single number representing a team's current strength. We use:
- **Initial ELO:** 1500 for all teams
- **K-factor:** 20 (update magnitude per match)
- **Home advantage:** +100 ELO points for expected-score calculation

ELO is updated after every match chronologically.

In [ ]:
K, HOME_ADV, INIT_ELO = 20, 100, 1500.0

def compute_elo(matches: pd.DataFrame) -> pd.DataFrame:
    df_ = matches.sort_values('match_date').reset_index(drop=True).copy()
    ratings = {}
    h_elos, a_elos = [], []
    for _, r in df_.iterrows():
        h, a = r.home_team, r.away_team
        he, ae = ratings.get(h, INIT_ELO), ratings.get(a, INIT_ELO)
        h_elos.append(he); a_elos.append(ae)
        exp_h = 1 / (1 + 10 ** ((ae - (he + HOME_ADV)) / 400))
        act_h = 1. if r.home_score > r.away_score else (0.5 if r.home_score == r.away_score else 0.)
        ratings[h] = he + K * (act_h - exp_h)
        ratings[a] = ae + K * ((1 - act_h) - (1 - exp_h))
    df_['home_elo_pre'] = h_elos
    df_['away_elo_pre'] = a_elos
    df_['ratings_snapshot'] = [dict(ratings)] * len(df_)  # for time-series
    return df_, ratings

matches_elo, final_ratings = compute_elo(df)

# Current top 20 ELO standings
top20 = pd.Series(final_ratings).sort_values(ascending=False).head(20).reset_index()
top20.columns = ['Team', 'ELO']
top20['Rank'] = range(1, 21)

fig = px.bar(top20, x='ELO', y='Team', orientation='h',
              text=top20['ELO'].round(0).astype(int),
              color='ELO', color_continuous_scale='Viridis',
              title='⚡ Current ELO Rankings — Top 20 National Teams')
fig.update_layout(height=550, paper_bgcolor='#0e1117', plot_bgcolor='#0e1117',
                   font=dict(color='white'), coloraxis_showscale=False,
                   yaxis=dict(categoryorder='total ascending'))
fig.update_traces(textposition='inside')
fig.show()

In [ ]:
# ELO history time-series for selected teams
TRACK_TEAMS = ['Brazil', 'Argentina', 'France', 'Germany', 'Spain',
                'England', 'Italy', 'Portugal', 'Netherlands', 'Morocco']

elo_history = []
running = {}
for _, r in matches_elo.iterrows():
    for side in ['home', 'away']:
        team = r[f'{side}_team']
        if team in TRACK_TEAMS:
            elo_history.append({'date': r.match_date, 'team': team,
                                  'elo': r[f'{side}_elo_pre']})

elo_df = pd.DataFrame(elo_history)

fig = px.line(elo_df, x='date', y='elo', color='team',
               title='📈 ELO Rating History — 2000 to Present',
               labels={'elo': 'ELO Rating', 'date': '', 'team': 'Nation'})
fig.update_layout(height=430, paper_bgcolor='#0e1117', plot_bgcolor='#0e1117',
                   font=dict(color='white'))
fig.show()

## 4. Feature Engineering <a id='4'></a>

We engineer features from a **team-perspective** view (two rows per match): each row represents one team's experience of the match.

| Feature Group | Features |
|---|---|
| ELO | `team_elo`, `opponent_elo`, `elo_diff` |
| Rolling form | `form_5`, `form_10`, `goals_for/against_avg_5/10` |
| Tournament form | `tournament_form_5`, `tournament_gf/ga_avg_5` |
| Head-to-head | `h2h_n`, `h2h_wins/draws/losses`, `h2h_gf/ga` |
| Context | `is_home`, `is_neutral`, `is_knockout`, `rest_days_diff`, `same_confederation` |
| Form diff | `form_diff_5/10` (team – opponent rolling form) |

In [ ]:
ROLLING_WINDOWS = [5, 10]
H2H_WINDOW = 10

CONFEDERATION_MAP = {
    'Brazil':'CONMEBOL','Argentina':'CONMEBOL','Uruguay':'CONMEBOL','Colombia':'CONMEBOL',
    'Chile':'CONMEBOL','Peru':'CONMEBOL','Ecuador':'CONMEBOL','Paraguay':'CONMEBOL',
    'Germany':'UEFA','France':'UEFA','Spain':'UEFA','Italy':'UEFA','England':'UEFA',
    'Netherlands':'UEFA','Portugal':'UEFA','Belgium':'UEFA','Croatia':'UEFA',
    'Switzerland':'UEFA','Poland':'UEFA','Denmark':'UEFA','Sweden':'UEFA',
    'Austria':'UEFA','Turkey':'UEFA','Ukraine':'UEFA','Czechia':'UEFA','Serbia':'UEFA',
    'Japan':'AFC','South Korea':'AFC','Australia':'AFC','Iran':'AFC','Saudi Arabia':'AFC',
    'Qatar':'AFC','Iraq':'AFC','Uzbekistan':'AFC',
    'United States':'CONCACAF','Mexico':'CONCACAF','Canada':'CONCACAF',
    'Costa Rica':'CONCACAF','Jamaica':'CONCACAF','Panama':'CONCACAF',
    'Morocco':'CAF','Nigeria':'CAF','Senegal':'CAF','Ivory Coast':'CAF','Egypt':'CAF',
    'Cameroon':'CAF','Ghana':'CAF','Algeria':'CAF','Tunisia':'CAF','South Africa':'CAF',
    'Mali':'CAF','DR Congo':'CAF',
    'New Zealand':'OFC',
}

def build_base(matches):
    rows = []
    for _, m in matches.iterrows():
        neutral = int(m.get('is_neutral', 0))
        t = str(m.get('tournament', '')).lower()
        if 'world cup' in t and 'qualif' not in t: tcat = 'world_cup'
        elif ('euro' in t or 'european' in t) and 'qualif' not in t: tcat = 'euro'
        elif 'qualif' in t: tcat = 'qualifier'
        elif 'friendly' in t: tcat = 'friendly'
        else: tcat = 'other'
        for team, opp, gf, ga, is_home in [
            (m.home_team, m.away_team, m.home_score, m.away_score, 1 if not neutral else 0),
            (m.away_team, m.home_team, m.away_score, m.home_score, 0)
        ]:
            gf, ga = int(gf), int(ga)
            rows.append({
                'match_id': m.match_id, 'match_date': m.match_date,
                'tournament': m.get('tournament',''), 'tcat': tcat,
                'team': team, 'opponent': opp,
                'goals_for': gf, 'goals_against': ga,
                'is_home': is_home, 'is_neutral': neutral,
                'stage': m.get('stage', 'group'),
                'is_knockout': int(m.get('stage','group') != 'group'),
                'outcome': 2 if gf > ga else (1 if gf == ga else 0),
                'points': 3 if gf > ga else (1 if gf == ga else 0),
            })
    return pd.DataFrame(rows).sort_values('match_date').reset_index(drop=True)


def rolling_features(df):
    out = df.sort_values(['team','match_date']).copy()
    grp = out.groupby('team', group_keys=False)
    for w in ROLLING_WINDOWS:
        out[f'form_{w}'] = grp['points'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum())
        out[f'gf_avg_{w}'] = grp['goals_for'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        out[f'ga_avg_{w}'] = grp['goals_against'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
    return out


def rest_days(df):
    out = df.sort_values(['team','match_date']).copy()
    out['rest'] = out.groupby('team')['match_date'].diff().dt.days.fillna(14).clip(1,365)
    opp_rest = (out[['match_id','team','rest']].drop_duplicates(['match_id','team'])
                .rename(columns={'team':'opponent','rest':'opp_rest'}))
    out = out.merge(opp_rest, on=['match_id','opponent'], how='left').fillna({'opp_rest':14})
    out['rest_diff'] = out['rest'] - out['opp_rest']
    return out


def tournament_form(df):
    out = df.sort_values(['team','tcat','match_date']).copy()
    grp = out.groupby(['team','tcat'], group_keys=False)
    out['t_form_5'] = grp['points'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
    out['t_gf_5'] = grp['goals_for'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    out['t_ga_5'] = grp['goals_against'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    return out


def h2h_features(base):
    pair_hist = defaultdict(lambda: deque(maxlen=H2H_WINDOW))
    lookup = {}
    for _, row in base.sort_values('match_date').iterrows():
        key = (row.match_id, row.team, row.opponent)
        if key not in lookup:
            prior = list(pair_hist[(row.team, row.opponent)])
            lookup[key] = {
                'h2h_n': len(prior),
                'h2h_wins': sum(p['o'] == 2 for p in prior),
                'h2h_draws': sum(p['o'] == 1 for p in prior),
                'h2h_losses': sum(p['o'] == 0 for p in prior),
                'h2h_gf': sum(p['gf'] for p in prior),
                'h2h_ga': sum(p['ga'] for p in prior),
            }
            pair_hist[(row.team, row.opponent)].append({'gf': row.goals_for, 'ga': row.goals_against, 'o': row.outcome})
            pair_hist[(row.opponent, row.team)].append({'gf': row.goals_against, 'ga': row.goals_for,
                                                         'o': 2 if row.outcome==0 else (0 if row.outcome==2 else 1)})
    records = [lookup.get((r.match_id, r.team, r.opponent), {'h2h_n':0,'h2h_wins':0,'h2h_draws':0,'h2h_losses':0,'h2h_gf':0,'h2h_ga':0})
               for _, r in base.iterrows()]
    return pd.concat([base.reset_index(drop=True), pd.DataFrame(records)], axis=1)


def add_elo_per_team(df, matches_elo_df):
    emap = matches_elo_df.set_index('match_id')[['home_team','away_team','home_elo_pre','away_elo_pre']]
    out = df.copy()
    def _elo(row, side):
        try:
            m = emap.loc[row.match_id]
            if row.team == m.home_team: return m.home_elo_pre if side == 'team' else m.away_elo_pre
            return m.away_elo_pre if side == 'team' else m.home_elo_pre
        except: return INIT_ELO
    out['team_elo'] = out.apply(lambda r: _elo(r, 'team'), axis=1)
    out['opp_elo'] = out.apply(lambda r: _elo(r, 'opp'), axis=1)
    out['elo_diff'] = out['team_elo'] - out['opp_elo']
    return out


def add_form_diff(df):
    out = df.copy()
    for w in ROLLING_WINDOWS:
        col = f'form_{w}'
        if col not in out.columns: continue
        opp_vals = (df[['match_id','team',col]].drop_duplicates(['match_id','team'])
                    .rename(columns={'team':'opponent', col: f'opp_{col}'}))
        out = out.merge(opp_vals, on=['match_id','opponent'], how='left')
        out[f'form_diff_{w}'] = out[col] - out[f'opp_{col}']
        out = out.drop(columns=[f'opp_{col}'])
    return out


# ── Run the pipeline ─────────────────────────────────────────────────────────
print('Building features... (this takes ~30s)')
base = build_base(df)
feat = rolling_features(base)
feat = rest_days(feat)
feat = tournament_form(feat)
feat = h2h_features(feat)
feat = add_elo_per_team(feat, matches_elo)
feat['confederation'] = feat['team'].map(CONFEDERATION_MAP).fillna('OTHER')
feat['opp_confederation'] = feat['opponent'].map(CONFEDERATION_MAP).fillna('OTHER')
feat['same_conf'] = (feat['confederation'] == feat['opp_confederation']).astype(int)
feat = add_form_diff(feat)

print(f'Feature matrix: {feat.shape[0]:,} rows × {feat.shape[1]} cols')
print(f'Outcome distribution: {feat.outcome.value_counts().to_dict()}')

In [ ]:
FEATURE_COLS = [
    'is_home','is_neutral','is_knockout','same_conf',
    'team_elo','opp_elo','elo_diff',
    'form_5','form_10','form_diff_5','form_diff_10',
    'gf_avg_5','ga_avg_5','gf_avg_10','ga_avg_10',
    't_form_5','t_gf_5','t_ga_5',
    'rest','rest_diff',
    'h2h_n','h2h_wins','h2h_draws','h2h_losses','h2h_gf','h2h_ga',
]
# Ensure all feature cols exist and are numeric
for c in FEATURE_COLS:
    if c not in feat.columns: feat[c] = 0.0
    feat[c] = pd.to_numeric(feat[c], errors='coerce').fillna(feat[c].median() if c in feat.columns else 0.0)

# Feature correlation heatmap (top 10 vs elo_diff)
corr = feat[FEATURE_COLS].corr()['elo_diff'].drop('elo_diff').abs().sort_values(ascending=False).head(10)
fig = px.bar(x=corr.values, y=corr.index, orientation='h',
              color=corr.values, color_continuous_scale='Blues',
              title='🔗 Feature Correlation with ELO Difference',
              labels={'x': '|Pearson r|', 'y': 'Feature'})
fig.update_layout(height=380, paper_bgcolor='#0e1117', plot_bgcolor='#0e1117',
                   font=dict(color='white'), coloraxis_showscale=False)
fig.show()

## 5. Match Outcome Model (XGBoost) <a id='5'></a>

**Temporal train/val/test split:**
- **Train** — all matches before 2018-06-01
- **Validation** — FIFA World Cup 2018 (64 matches)
- **Test (held-out)** — FIFA World Cup 2022 (64 matches)

This is the **honest** approach: the model never sees future World Cup matches during training.

In [ ]:
feat['match_date'] = pd.to_datetime(feat['match_date'])

is_wc = feat['tournament'].str.lower().str.contains('fifa world cup', na=False)
is_wc18 = is_wc & (feat.match_date >= '2018-06-01') & (feat.match_date <= '2018-07-31')
is_wc22 = is_wc & (feat.match_date >= '2022-11-01') & (feat.match_date <= '2022-12-31')

train = feat[~is_wc18 & ~is_wc22]
val   = feat[is_wc18]
test  = feat[is_wc22]

usable_cols = [c for c in FEATURE_COLS if feat[c].notna().any() and feat[c].nunique() > 1]

X_train, y_train = train[usable_cols], train['outcome']
X_val,   y_val   = val[usable_cols],   val['outcome']
X_test,  y_test  = test[usable_cols],  test['outcome']

print(f'Train: {len(train):,} | Val (WC 2018): {len(val)} | Test (WC 2022): {len(test)}')
print(f'Features used: {len(usable_cols)}')

In [ ]:
sample_w = compute_sample_weight('balanced', y_train)

xgb_pipe = Pipeline([
    ('imp',   SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('model', XGBClassifier(
        objective='multi:softprob', num_class=3,
        eval_metric='mlogloss',
        max_depth=5, learning_rate=0.08, n_estimators=200,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        verbosity=0,
    ))
])

print('Training XGBoost...')
xgb_pipe.fit(X_train, y_train, model__sample_weight=sample_w)

# Calibrate on 2018 WC
from sklearn.frozen import FrozenEstimator
calibrated = CalibratedClassifierCV(FrozenEstimator(xgb_pipe), method='sigmoid', cv='prefit')
calibrated.fit(X_val, y_val)

# Evaluate
def eval_model(name, y_true, proba):
    acc  = accuracy_score(y_true, proba.argmax(axis=1))
    ll   = log_loss(y_true, proba, labels=[0,1,2])
    print(f'  {name}: accuracy={acc:.3f}, log_loss={ll:.3f}')
    return acc, ll

print('\n── Validation (2018 WC):')
val_acc, val_ll = eval_model('Calibrated XGB', y_val.values, calibrated.predict_proba(X_val))

print('\n── Test (2022 WC — held out):')
test_proba = calibrated.predict_proba(X_test)
test_acc, test_ll = eval_model('Calibrated XGB', y_test.values, test_proba)

hc_mask = test_proba.max(axis=1) >= 0.55
hc_acc = accuracy_score(y_test.values[hc_mask], test_proba.argmax(axis=1)[hc_mask]) if hc_mask.sum() else 0
print(f'  High-confidence (≥55%): {hc_acc:.3f} on {hc_mask.sum()} matches')

In [ ]:
# SHAP feature importance
xgb_model = xgb_pipe.named_steps['model']
imp_df = pd.DataFrame({'feature': usable_cols,
                        'importance': xgb_model.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=True).tail(15)

FEAT_LABELS = {
    'is_home':'Home advantage','elo_diff':'ELO rating advantage','team_elo':'Team ELO',
    'opp_elo':'Opponent ELO','form_5':'Recent form (last 5)','form_10':'Form (last 10)',
    'form_diff_5':'Form advantage (5)','form_diff_10':'Form advantage (10)',
    'gf_avg_5':'Goals scored avg (5)','ga_avg_5':'Goals conceded avg (5)',
    'gf_avg_10':'Goals scored avg (10)','ga_avg_10':'Goals conceded avg (10)',
    't_form_5':'Tournament form (5)','rest':'Days since last match',
    'rest_diff':'Rest advantage','h2h_n':'H2H history count',
    'h2h_wins':'H2H wins','same_conf':'Same confederation',
    'is_neutral':'Neutral venue','is_knockout':'Knockout stage',
}
imp_df['label'] = imp_df['feature'].map(FEAT_LABELS).fillna(imp_df['feature'])

fig = px.bar(imp_df, x='importance', y='label', orientation='h',
              color='importance', color_continuous_scale='Plasma',
              title='🔍 XGBoost Feature Importance — Top 15')
fig.update_layout(height=450, paper_bgcolor='#0e1117', plot_bgcolor='#0e1117',
                   font=dict(color='white'), coloraxis_showscale=False)
fig.show()

In [ ]:
# Confusion matrix on 2022 WC test set
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test.values, test_proba.argmax(axis=1))
labels = ['Loss', 'Draw', 'Win']

fig, ax = plt.subplots(figsize=(6,4), facecolor='#0e1117')
ax.set_facecolor('#0e1117')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
             xticklabels=labels, yticklabels=labels,
             linewidths=0.5, linecolor='#333')
ax.set_xlabel('Predicted', color='white')
ax.set_ylabel('Actual', color='white')
ax.set_title('Confusion Matrix — 2022 WC Test Set', color='white', pad=12)
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()

## 6. WC 2026 Group Stage Predictions <a id='6'></a>

> ⚠️ The official WC 2026 group draw has not yet taken place. The groups below are **illustrative placeholders** based on likely-qualified nations. Replace with official groups once announced.

WC 2026 will feature **48 teams in 12 groups of 4**, played across USA, Canada, and Mexico.

In [ ]:
# WC 2026 sample groups (placeholder — replace after official draw)
WC2026_GROUPS = {
    'A': ['United States', 'Morocco', 'Uruguay', 'Serbia'],
    'B': ['Mexico', 'Germany', 'Japan', 'Ivory Coast'],
    'C': ['Canada', 'France', 'Argentina', 'Cameroon'],
    'D': ['Spain', 'Brazil', 'South Korea', 'Australia'],
    'E': ['England', 'Netherlands', 'Colombia', 'Senegal'],
    'F': ['Portugal', 'Belgium', 'Ecuador', 'Ghana'],
    'G': ['Italy', 'Croatia', 'Chile', 'Nigeria'],
    'H': ['Denmark', 'Switzerland', 'Saudi Arabia', 'Algeria'],
    'I': ['Austria', 'Poland', 'Iran', 'Tunisia'],
    'J': ['Turkey', 'Ukraine', 'Qatar', 'New Zealand'],
    'K': ['Scotland', 'Panama', 'Iraq', 'Egypt'],
    'L': ['Costa Rica', 'Jamaica', 'Uzbekistan', 'Paraguay'],
}

# Get most recent feature snapshot per team
latest = feat.sort_values('match_date').groupby('team').tail(1).set_index('team')

def get_team_features(team):
    if team in latest.index:
        row = {c: latest.loc[team, c] if c in latest.columns and pd.notna(latest.loc[team, c]) else 0.0
               for c in usable_cols}
    else:
        row = {c: 0.0 for c in usable_cols}
    row['team_elo'] = final_ratings.get(team, INIT_ELO)
    return row


def predict_match(team_a, team_b, stage='group', is_home=1):
    a_feats = get_team_features(team_a)
    b_feats = get_team_features(team_b)
    row = dict(a_feats)
    row['opp_elo'] = b_feats.get('team_elo', INIT_ELO)
    row['elo_diff'] = row['team_elo'] - row['opp_elo']
    for w in ROLLING_WINDOWS:
        f_col = f'form_{w}'
        if f_col in row and f_col in b_feats:
            row[f'form_diff_{w}'] = row.get(f_col, 0) - b_feats.get(f_col, 0)
    row['is_home'] = is_home
    row['is_neutral'] = int(is_home == 0)
    row['is_knockout'] = int(stage != 'group')
    a_conf = CONFEDERATION_MAP.get(team_a, 'OTHER')
    b_conf = CONFEDERATION_MAP.get(team_b, 'OTHER')
    row['same_conf'] = int(a_conf == b_conf)
    X = pd.DataFrame([{c: row.get(c, 0.0) for c in usable_cols}])
    proba = calibrated.predict_proba(X)[0]
    return {'win': float(proba[2]), 'draw': float(proba[1]), 'loss': float(proba[0])}


# Predict all group fixtures and simulate group standings
group_results = []
for group, teams in WC2026_GROUPS.items():
    for i, ta in enumerate(teams):
        for tb in teams[i+1:]:
            p = predict_match(ta, tb, is_home=0)  # neutral venue
            group_results.append({
                'group': group, 'team_a': ta, 'team_b': tb,
                'p_a_win': p['win'], 'p_draw': p['draw'], 'p_b_win': p['loss']
            })

fixtures = pd.DataFrame(group_results)
print(f'Total group stage fixtures: {len(fixtures)}')
fixtures.head(6)

In [ ]:
# Simulate group standings (expected points via probabilities)
team_ep = defaultdict(lambda: {'group':'','pts':0.0,'gf':0.0,'wins':0.0})
for _, grp_teams in WC2026_GROUPS.items():
    for t in grp_teams:
        team_ep[t]['group'] = _

for _, row in fixtures.iterrows():
    ta, tb = row.team_a, row.team_b
    team_ep[ta]['pts'] += row.p_a_win * 3 + row.p_draw
    team_ep[tb]['pts'] += row.p_b_win * 3 + row.p_draw
    team_ep[ta]['wins'] += row.p_a_win
    team_ep[tb]['wins'] += row.p_b_win

standings = pd.DataFrame([
    {'Group': v['group'], 'Team': k,
     'Exp. Points': round(v['pts'], 2),
     'ELO': round(final_ratings.get(k, INIT_ELO), 0),
     'Exp. Wins': round(v['wins'], 2)}
    for k, v in team_ep.items()
]).sort_values(['Group','Exp. Points'], ascending=[True, False]).reset_index(drop=True)

# Mark predicted qualifiers (top 2 per group)
standings['Pos'] = standings.groupby('Group').cumcount() + 1
standings['🏆 Qualifies'] = standings['Pos'] <= 2

# Show top contenders
top_teams = standings.nlargest(16, 'Exp. Points')[['Group','Team','Exp. Points','ELO','🏆 Qualifies']]
print('Top 16 teams by expected group points:')
display(top_teams)

In [ ]:
# Predicted group winners bar chart
group_winners = standings[standings.Pos == 1].sort_values('Exp. Points', ascending=False)

fig = px.bar(group_winners, x='Team', y='Exp. Points',
              color='Exp. Points', color_continuous_scale='Viridis',
              text='Group', title='🥇 Predicted WC 2026 Group Winners (by Expected Points)')
fig.update_layout(height=420, paper_bgcolor='#0e1117', plot_bgcolor='#0e1117',
                   font=dict(color='white'), coloraxis_showscale=False)
fig.show()

## 7. xG-Based Match Simulation <a id='7'></a>

Using **pre-aggregated StatsBomb xG** (from the open data match files) to estimate each team's attacking/defensive xG rate.  
We then run a **Poisson Monte Carlo** simulation (10,000 games) to produce scoreline probabilities.

In [ ]:
# Download StatsBomb open data — matches only (no large event files)
SB_RAW = 'https://raw.githubusercontent.com/statsbomb/open-data/master/data'
WC_COMP_ID = 43  # FIFA World Cup

# Competition index
comps = requests.get(f'{SB_RAW}/competitions.json', timeout=30).json()
wc_seasons = [c for c in comps if c['competition_id'] == WC_COMP_ID]
print(f'WC seasons available: {[c["season_name"] for c in wc_seasons]}')

# Load matches for each WC season and collect pre-computed xG
sb_rows = []
for season in wc_seasons:
    sid = season['season_id']
    url = f'{SB_RAW}/matches/{WC_COMP_ID}/{sid}.json'
    try:
        matches_json = requests.get(url, timeout=30).json()
        for m in matches_json:
            ht = norm(m.get('home_team', {}).get('home_team_name', ''))
            at = norm(m.get('away_team', {}).get('away_team_name', ''))
            # StatsBomb match-level metadata includes metadata_version xG
            meta = m.get('metadata', {})
            sb_rows.append({
                'season': season['season_name'],
                'home_team': ht, 'away_team': at,
                'home_score': m.get('home_score', 0),
                'away_score': m.get('away_score', 0),
                'match_date': m.get('match_date',''),
            })
    except Exception as e:
        print(f'  Skip season {sid}: {e}')

sb_df = pd.DataFrame(sb_rows)
print(f'StatsBomb WC matches loaded: {len(sb_df)}')
sb_df.head(3)

In [ ]:
# Use goals scored as xG proxy (since we're using pre-aggregated matches.json)
# Compute per-team attacking (goals for) and defensive (goals against) rates
xg_long = pd.concat([
    sb_df.assign(team=sb_df.home_team, xg_for=sb_df.home_score, xg_against=sb_df.away_score),
    sb_df.assign(team=sb_df.away_team, xg_for=sb_df.away_score, xg_against=sb_df.home_score)
])
team_xg = (xg_long.groupby('team')
            .agg(xg_for_rate=('xg_for','mean'), xg_against_rate=('xg_against','mean'),
                  n_matches=('xg_for','count'))
            .reset_index())

# Default for teams not in StatsBomb
DEFAULT_XG = {'xg_for_rate': 1.2, 'xg_against_rate': 1.2}

def simulate_match_xg(team_a: str, team_b: str, n: int = 10_000) -> dict:
    ra = team_xg[team_xg.team == team_a].iloc[0].to_dict() if team_a in team_xg.team.values else DEFAULT_XG
    rb = team_xg[team_xg.team == team_b].iloc[0].to_dict() if team_b in team_xg.team.values else DEFAULT_XG
    lam_a = max(0.3, (ra['xg_for_rate'] + rb['xg_against_rate']) / 2)
    lam_b = max(0.3, (rb['xg_for_rate'] + ra['xg_against_rate']) / 2)
    np.random.seed(42)
    ga = np.random.poisson(lam_a, n)
    gb = np.random.poisson(lam_b, n)
    scorelines = defaultdict(int)
    for a, b in zip(ga, gb):
        scorelines[(int(a), int(b))] += 1
    top = sorted(scorelines.items(), key=lambda x: x[1], reverse=True)[:8]
    return {
        'team_a': team_a, 'team_b': team_b,
        'xg_a': round(lam_a, 2), 'xg_b': round(lam_b, 2),
        'win_a': float((ga > gb).mean()),
        'draw':  float((ga == gb).mean()),
        'win_b': float((ga < gb).mean()),
        'top_scorelines': [{'score': f'{s[0]}-{s[1]}', 'prob': round(c/n, 3)} for s, c in top],
    }

# Example: Argentina vs France (WC 2022 Final rematch)
result = simulate_match_xg('Argentina', 'France')
print(f"\n⚽ Argentina vs France (xG Monte Carlo, 10k sims)")
print(f"  Expected xG → Argentina: {result['xg_a']:.2f} | France: {result['xg_b']:.2f}")
print(f"  Win A: {result['win_a']:.1%} | Draw: {result['draw']:.1%} | Win B: {result['win_b']:.1%}")
print("\n  Top scorelines:")
for s in result['top_scorelines'][:5]:
    print(f"    {s['score']} → {s['prob']:.1%}")

In [ ]:
# Scoreline heatmap for a marquee fixture
FIXTURE_A, FIXTURE_B = 'Brazil', 'Argentina'
res = simulate_match_xg(FIXTURE_A, FIXTURE_B, n=50_000)

max_goals = 6
grid = np.zeros((max_goals+1, max_goals+1))
np.random.seed(42)
ga = np.random.poisson(res['xg_a'], 50000)
gb = np.random.poisson(res['xg_b'], 50000)
for a, b in zip(ga, gb):
    if a <= max_goals and b <= max_goals:
        grid[a, b] += 1
grid = grid / grid.sum() * 100

fig, ax = plt.subplots(figsize=(7,5), facecolor='#0e1117')
ax.set_facecolor('#0e1117')
im = ax.imshow(grid, cmap='YlOrRd', aspect='auto')
for i in range(max_goals+1):
    for j in range(max_goals+1):
        ax.text(j, i, f'{grid[i,j]:.1f}%', ha='center', va='center',
                 color='black' if grid[i,j] > 5 else 'white', fontsize=8)
ax.set_xticks(range(max_goals+1)); ax.set_xticklabels(range(max_goals+1), color='white')
ax.set_yticks(range(max_goals+1)); ax.set_yticklabels(range(max_goals+1), color='white')
ax.set_xlabel(FIXTURE_B, color='white', fontsize=12)
ax.set_ylabel(FIXTURE_A, color='white', fontsize=12)
ax.set_title(f'🎯 Scoreline Probability Heatmap: {FIXTURE_A} vs {FIXTURE_B}', color='white', pad=12)
plt.colorbar(im, ax=ax, label='Probability (%)')
plt.tight_layout()
plt.show()

## 8. Penalty Shootout Simulator <a id='8'></a>

Monte Carlo simulation of FIFA-rules penalty shootouts.  
- **Base conversion rate:** 75.7% (historical average)
- **Keeper save modifier:** elite keepers can add 6–12% save probability
- **Pressure decay:** conversion probability drops in sudden-death rounds
- **First-kick advantage:** tiny boost (~1.2%) for the team kicking first

In [ ]:
BASE_CONV = 0.757
PRESSURE_DECAY = 0.03
FIRST_KICK_ADV = 0.012

KEEPER_PRESETS = {
    'Average keeper':           0.00,
    'Emi Martínez (ARG)':       0.12,
    'Dominik Livaković (CRO)':  0.11,
    'Yann Sommer (SUI)':        0.10,
    'Jordan Pickford (ENG)':    0.10,
    'Thibaut Courtois (BEL)':   0.09,
    'Manuel Neuer (GER)':       0.08,
    'Hugo Lloris (FRA)':        0.07,
    'Ederson (BRA)':            0.06,
}

def _conv_prob(skill, keeper_save, round_num, first):
    p = BASE_CONV + skill - keeper_save
    if round_num > 5: p -= PRESSURE_DECAY * (round_num - 5)
    if first: p += FIRST_KICK_ADV
    return float(np.clip(p, 0.30, 0.98))

def simulate_shootout(a_skills, b_skills, a_keeper=0.0, b_keeper=0.0, n=10_000, seed=42):
    rng = np.random.default_rng(seed)
    wins_a = wins_b = 0
    total_rounds = 0
    sample_log = []
    for i in range(n):
        first = 'a' if i % 2 == 0 else 'b'
        sa, sb, log = 0, 0, []
        for r in range(1, 31):
            for is_first, team in [(True, first), (False, 'b' if first == 'a' else 'a')]:
                skills = a_skills if team == 'a' else b_skills
                ks = b_keeper if team == 'a' else a_keeper
                prob = _conv_prob(skills[(r-1) % len(skills)], ks, r, is_first)
                scored = bool(rng.random() < prob)
                if team == 'a' and scored: sa += 1
                elif team == 'b' and scored: sb += 1
                log.append({'r': r, 'team': team, 'scored': scored, 'sa': sa, 'sb': sb})
            rem = max(0, 5 - r)
            if r <= 5 and (sa > sb + rem or sb > sa + rem): break
            if r > 5 and sa != sb: break
        winner = 'a' if sa >= sb else 'b'
        if winner == 'a': wins_a += 1
        else: wins_b += 1
        total_rounds += r
        if i == 0: sample_log = log
    return {'win_a': wins_a/n, 'win_b': wins_b/n, 'avg_rounds': total_rounds/n, 'log': sample_log}

# Example: England vs Argentina (classic shootout matchup)
result = simulate_shootout(
    a_skills=[0.05, 0.08, 0.05, 0.05, 0.08],  # England takers
    b_skills=[0.10, 0.10, 0.08, 0.08, 0.10],  # Argentina takers
    a_keeper=KEEPER_PRESETS['Jordan Pickford (ENG)'],
    b_keeper=KEEPER_PRESETS['Emi Martínez (ARG)'],
    n=50_000
)
print(f'🥅 England vs Argentina — Penalty Shootout (50k sims)')
print(f'  England wins:   {result["win_a"]:.1%}')
print(f'  Argentina wins: {result["win_b"]:.1%}')
print(f'  Average rounds: {result["avg_rounds"]:.1f}')

In [ ]:
# Visualise sample shootout kick log
log = result['log']
log_df = pd.DataFrame(log)

fig, ax = plt.subplots(figsize=(10, 3.5), facecolor='#0e1117')
ax.set_facecolor('#0e1117')

for row in log_df.itertuples():
    color = '#00d4aa' if row.team == 'a' else '#ff6b6b'
    marker = '⚽' if row.scored else '✗'
    y = 0.7 if row.team == 'a' else 0.3
    ax.text(row.Index, y, marker, fontsize=16, ha='center', va='center',
             color=color, weight='bold')
    ax.text(row.Index, 0.5, f"{row.sa}-{row.sb}", fontsize=7,
             ha='center', va='center', color='white')

ax.set_ylim(0, 1)
ax.set_xlim(-0.5, len(log_df) - 0.5)
ax.set_yticks([0.3, 0.7])
ax.set_yticklabels(['Argentina', 'England'], color='white')
ax.tick_params(colors='white')
ax.set_xlabel('Kick #', color='white')
ax.set_title('🎯 Sample Shootout Kick-by-Kick (⚽ = scored, ✗ = missed)', color='white', pad=10)
plt.tight_layout()
plt.show()

## 9. Key Findings <a id='9'></a>

### What the model tells us:

| Finding | Detail |
|---------|--------|
| **ELO is king** | `elo_diff` is the single strongest predictor. A 200-point gap implies ~70% win probability for the stronger side |
| **Home advantage is real but modest** | ~5–8% boost in win probability for non-neutral venues |
| **Recent form matters more than H2H** | Rolling 5-match form outranks head-to-head history in feature importance |
| **Draws are hardest to predict** | The model under-predicts draws; ~26% of WC matches end level but the model assigns highest probability of Draw far less often |
| **Three-class accuracy ceiling** | ~60–62% on WC test sets is strong — random baseline is 33%, bookmaker-implied is ~63% |
| **Penalty shootouts are close to 50-50** | Even with elite keepers, win probability rarely exceeds 60-40 |

### WC 2026 early forecasts (based on placeholder groups):
- 🇦🇷 **Argentina** — strongest ELO + highest expected points in their group
- 🇫🇷 **France** — consistent top performer across form metrics
- 🇧🇷 **Brazil** — high attacking xG rate historically
- 🇩🇪 **Germany** — strong tournament-specific form
- 🇲🇦 **Morocco** — most improved ELO trajectory post-2022

---

### Further development ideas:
1. Add **Transfermarkt squad market values** (see repo `src/transfermarkt_client.py`)
2. Add **FBref player aggregates** (goals/90, xG/90) when squad data is available
3. Train **separate models** for knockout vs group stages
4. Add **Poisson regression** goal model alongside the classification model
5. Run a **full tournament Monte Carlo** (bracket simulation, 100k iterations) for title-win probabilities

---

> 🔗 **Full Streamlit app** (live dashboard with SHAP explanations, xG engine, knockout bracket):  
> GitHub: [shubhamindulkar/fifa-world-cup-analysis](https://github.com/Thesilentprogramer/fifa-world-cup-analysis)
